Dieses Skript definiert einen einfachen VAE, der auf dem MNIST-Datensatz trainiert wird. Es verwendet die MLPRegressor-Klasse von Scikit-learn, um den Encoder und Decoder zu definieren. Der Encoder wird auf dem Eingabe-Datensatz trainiert, um die mittlere und logarithmische Varianz der Latent-Variable zu schätzen. Der Decoder wird dann auf den geschätzten Mittelwert der Latent-Variable trainiert, um die ursprünglichen Eingabedaten zu rekonstruieren. Der VAE wird durch den Aufruf der VAE-Funktion trainiert, die eine Epoche über den Datensatz läuft und für jeden Batch eine Rekonstruktion der Eingabe durchführt. Die reparameterize-Methode aus dem vorherigen Beispiel wird hier durch das Sampling von Normalverteilungen und die Berechnung der rekonstruierten Eingabe durch den Decoder ersetzt.

In [ ]:
import numpy as np
from sklearn.neural_network import MLPRegressor

# Hyperparameters
latent_dim = 2
input_dim = 784
hidden_dim = 256
batch_size = 128
epochs = 10

# Load MNIST dataset
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0

# Define encoder model
encoder = MLPRegressor(hidden_layer_sizes=(hidden_dim, hidden_dim, latent_dim * 2), activation="relu")

# Train encoder
encoder.fit(x_train, x_train)

# Extract mean and log variance from encoder
z_mean, z_log_var = encoder.predict(x_train)[:,:latent_dim], encoder.predict(x_train)[:,latent_dim:]

# Define decoder model
decoder = MLPRegressor(hidden_layer_sizes=(hidden_dim, hidden_dim, input_dim), activation="relu")

# Train decoder
decoder.fit(z_mean, x_train)

# Define VAE model
def VAE(x):
    # Encode input
    z_mean, z_log_var = encoder.predict(x)[:,:latent_dim], encoder.predict(x)[:,latent_dim:]
    
    # Sample latent variables
    eps = np.random.normal(size=(batch_size, latent_dim))
    z = z_mean + np.exp(0.5 * z_log_var) * eps
    
    # Decode latent variables
    reconstructed = decoder.predict(z)
    
    return reconstructed

# Train VAE model
for epoch in range(epochs):
    for i in range(0, len(x_train), batch_size):
        x_batch = x_train[i:i+batch_size]
        VAE(x_batch)
